###### <font color='blue'> Random forest classification model </font>

####  <font color='blue'> Sabah Sabaghy 15 July 2026

"""

Satellite Image Classification using Random Forest (Parallel Optimised Version)

This workflow classifies multi-band satellite image tiles into land cover classes
using a Random Forest model trained from ground truth data. The script reads
satellite imagery, training data, and masks, predicts land cover classes, and
exports classified maps and class probability layers.

The workflow has been further developed and optimised to improve performance
through faster multi-band image reading and parallel processing of image tiles.
These enhancements significantly reduce processing time for large datasets while
preserving consistency with the original classification workflow.

Note: Satellite, training, and mask data tiles must share the same filename
and tile ID for correct processing.

"""

## Read input data, including ground truth data and images

In [ ]:
from osgeo import gdal, gdal_array
import os
import glob
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# Tell GDAL to throw Python exceptions, and register all drivers
gdal.UseExceptions()
gdal.AllRegister()

In [ ]:
# read satellite image tile
dir_sat = r"..."

# read training data tiles
dir_train = r"..."

count = 0
for path in os.listdir(dir_sat):
    if os.path.isfile(os.path.join(dir_sat, path)):
        count += 1
        
print('Number of tiles to process: ', count)

# read tiles and save them in a dataframe

with rasterio.open(r"...\Tile.tif") as data:
    bands = data.count

df_img = pd.DataFrame(dtype='float', columns=['band' + str(x) for x in range(1,bands+1)])
df_train = pd.DataFrame(dtype='float', columns=['Training_Data'])

for filename in glob.iglob(f'{dir_sat}//*'):
    
    with rasterio.open(filename) as img_ds:
        tmp = np.zeros((img_ds.height*img_ds.width, img_ds.count))
        for b in range(img_ds.count):
            tmp[:, b] = img_ds.read(b+1).flatten()
        df_tmp = pd.DataFrame(tmp, columns=['band' + str(x) for x in range(1,b+2)])
    df_img = pd.concat([df_img, df_tmp], axis = 0)
    
for filename in glob.iglob(f'{dir_train}//*'):
    
    with rasterio.open(filename) as roi:
        tmp_roi = np.zeros((roi.height*roi.width, roi.count))
        tmp_roi[:, 0] = roi.read(1).flatten()
        df_roi = pd.DataFrame(tmp_roi, columns=['Training_Data'])
        df_roi['Training_Data'] = df_roi['Training_Data'].replace(roi.nodatavals[0], np.nan)
    df_train = pd.concat([df_train, df_roi], axis = 0)

df = pd.concat([df_img, df_train], axis=1)
df = df[df['Training_Data'].notna()]

## Using Skicit-learn to split ground truth and image data into training and testing sets

In [ ]:
%%time
from sklearn.model_selection import train_test_split

features = df.drop(columns=['Training_Data'], axis=1) 
labels = df['Training_Data']

# Split the data into training and testing sets
# train_test_split: Allowed inputs are lists, numpy arrays, scipy-sparse matrices or pandas dataframes.
train_features, test_features, train_labels, test_labels = train_test_split(features, labels, test_size = 0.30, random_state = 42)

## Make sure the split of data is correct

In [ ]:
print('Training Features Shape:', train_features.shape)
print('Training Labels Shape:', train_labels.shape)
print('Testing Features Shape:', test_features.shape)
print('Testing Labels Shape:', test_labels.shape)

## Model Tuning

### Initialize a Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize our model
rf = RandomForestClassifier(random_state=42)

In [ ]:
# select hyperparameters (hyperparameters are sorted based on their importance; number od estimators/trees is the most important parameter)

# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 5)]
# Number of features to consider at every split
max_features = ['log2', 'sqrt']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(5, 25, num = 5)]
# Minimum number of samples required to split a node
min_samples_split = [2,5,10,15]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1,2, 4, 10]
bootstrap = [True, False]
# oob_score = [True, False]

hyperF = dict(n_estimators = n_estimators, max_depth = max_depth,  
              min_samples_split = min_samples_split, min_samples_leaf = min_samples_leaf,
             max_features = max_features, bootstrap = bootstrap) #, oob_score = oob_score

### Finding best machine learning model hyperparameters
#### <font color='blue'>**Randomized Search CV:** </font>
Random Search sets up a grid of hyperparameter values and selects random combinations to train the model and score. This allows you to explicitly control the number of parameter combinations that are attempted. The number of search iterations is set based on time or resources.<br>
<font color='green'> **Pros**: reduced overfitting problem, more accurate long term results </font>

In [ ]:
%%time
# When having limitted resources: conduct a Randomized Search CV
from sklearn.model_selection import RandomizedSearchCV
import dask_ml.model_selection as dcv

randomF = dcv.RandomizedSearchCV(rf, hyperF, n_iter=20, n_jobs=-1, cv=3, random_state=42)

# Train Random Forest Model
bestF = randomF.fit(train_features, train_labels)
print("The mean accuracy of the model is:", bestF.score(test_features, test_labels))

In [ ]:
# Save the trained random forest model
import joblib

joblib.dump(bestF, "./Mallee_RF_model_31bands_20m_test.joblib")

In [ ]:
# get the best estimator that was chosen by the search
bestF.best_estimator_

In [ ]:
# check the cv results
import pandas as pd
pd.DataFrame(bestF.cv_results_)

## Evaluate model performance

In [ ]:
def evaluate(model, test_features, test_labels):
    # Determine Performance Metrics 
    predictions = model.predict(test_features)
    
    # Get the Confusion Matrix, Classification report and ACcuracy
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

    print('Confusion Matrix:', confusion_matrix(test_labels, predictions))
    print (classification_report(test_labels, predictions))
    print("Accuracy:", accuracy_score(test_labels, predictions) * 100) 
    
    return predictions

In [ ]:
evalu = evaluate(bestF, test_features, test_labels)

## Visualize confusion matrix 

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


labels_1 = {'1':'Bare', '3':'Cereals', '4':'Deciduous fruit tree'}
label_temp = [1, 3, 4]


labels_list = [labels_1[key] for key in labels_1]

cm = confusion_matrix(test_labels, evalu, labels=label_temp)
# print(cm)

cmd = ConfusionMatrixDisplay(cm, display_labels=labels_1.values())
cmd.plot(xticks_rotation='vertical', cmap='Greens')

## Find the importance of each satellite image band

In [ ]:
import math
bands = list(range(31))

for b, imp in zip(bands, bestF.best_estimator_.feature_importances_):
    print('Band {b} importance: {imp}'.format(b = b, imp = round(imp, 2)))

## Classify tiles from a satellite image based on a pre-trained Random Forest model

In [ ]:
%%time
# Assign directory where input imagery files (tiles) to be classified are located
dir_img = r"..."
dir_mask = r'...'

# Assign directories to save output rasters
dir_class = r"..."        
dir_prob = r"..."

# Count number of files to classify
count = 0
for path in os.listdir(dir_img):
    if os.path.isfile(os.path.join(dir_img, path)):
        count += 1
print("Images to process = " + str(count))

In [ ]:
%%time
import joblib
# Import saved Random Forest model
loaded_rf = joblib.load(".\\model.joblib")
# Enter model name to be used in output file names
modelname = "model"

# Iterate through the files in the assigned directory
with rasterio.open(r"...\Tile.tif") as data:
    bands = data.count

df_img = pd.DataFrame(dtype='float', columns=['band' + str(x) for x in range(1,bands+2)])

# find a file
def find(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)

for filename in glob.iglob(f'{dir_img}//*'):
    
    # Open image file
    with rasterio.open(filename) as img:
        arr = np.zeros((img.height*img.width, img.count))
        for b in range(img.count):
            arr[:, b] = img.read(b+1).flatten()
        df_arr = pd.DataFrame(arr, columns=['band' + str(x) for x in range(1,b+2)])
        
        # find a relevant mask
        name = 'Raster_Mask_' + filename[-11:]
        filename_msk = find(name, dir_mask)
        
        # open mask file
        with rasterio.open(filename_msk) as msk:
            mask = np.zeros((msk.height*msk.width, msk.count))
            mask[:, 0] = msk.read(1).flatten()
        df_msk = pd.DataFrame(mask, columns=['mask'])
        # replace nodata with nan 
        df_msk['mask'] = df_msk['mask'].replace(msk.nodatavals[0], np.nan)
        
        df_img = pd.concat([df_arr, df_msk], axis=1)

        # mask out nan data
        df_img.dropna(inplace=True)
        
        ini_data = df_img.drop(columns=['mask'], axis=1) 
        # get the index of remaining rows
        ind = ini_data.index.tolist()
        
        if ini_data.shape[0] !=0:
            # Apply the Random Forest model and predict a class - for each pixel
            class_prediction = loaded_rf.predict(ini_data)
            # Concatenate outputs to the mask image so that value of predictions can get attached to relevant pixels when reshaped
            df_class = pd.DataFrame(class_prediction, columns=['predictions'], index=ind)
            df_out = pd.concat([df_msk, df_class], axis = 1).drop(columns=['mask'])
            out = df_out['predictions'].tolist()
            
        else:
            out = np.repeat([[np.nan]]*msk.height, msk.width, axis=1).tolist()
            
        # Reshape the classified raster
        class_pred = np.array(out).reshape(img.shape)
        
        # Create output file name
        imgname = os.path.basename(filename).split('.')[0]
        path = dir_class + imgname + "_" + modelname + "_allclasses.tif"
        
        # get image metadata
        kwargs = msk.meta
        
        # save output rasters
        with rasterio.open(path, 'w', **kwargs) as dst:
            dst.write_band(1, class_pred)
    
        print(path)

## Calculate probaility of each class

In [ ]:
%%time
import joblib
# Import saved Random Forest model
loaded_rf = joblib.load(".\\model.joblib")
# Enter model name to be used in output file names
modelname = "model"

# Iterate through the files in the assigned directory
with rasterio.open(r"...\Tile.tif") as data:
    bands = data.count

df_img = pd.DataFrame(dtype='float', columns=['band' + str(x) for x in range(1,bands+2)])

# find a file
def find(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)

for filename in glob.iglob(f'{dir_img}//*'):
    
    # Open image file
    with rasterio.open(filename) as img:
        arr = np.zeros((img.height*img.width, img.count))
        for b in range(img.count):
            arr[:, b] = img.read(b+1).flatten()
        df_arr = pd.DataFrame(arr, columns=['band' + str(x) for x in range(1,b+2)])
        
        # find a relevant mask
        name = 'Raster_Mask_' + filename[-11:]
        filename_msk = find(name, dir_mask)
        
        # open mask file
        with rasterio.open(filename_msk) as msk:
            mask = np.zeros((msk.height*msk.width, msk.count))
            mask[:, 0] = msk.read(1).flatten()
        df_msk = pd.DataFrame(mask, columns=['mask'])
        # replace nodata with nan 
        df_msk['mask'] = df_msk['mask'].replace(msk.nodatavals[0], np.nan)
        
        df_img = pd.concat([df_arr, df_msk], axis=1)

        # mask out nan data
        df_img.dropna(inplace=True)
        
        ini_data = df_img.drop(columns=['mask'], axis=1) 
        # get the index of remaining rows
        ind = ini_data.index.tolist()
        
        if ini_data.shape[0] !=0:
            
            # Calculate the probability of the class being alligator weed (class 3) - for each pixel
            class_probability = np.max(loaded_rf.predict_proba(ini_data), axis=1)
            df_prob = pd.DataFrame(class_probability, columns=['probability'], index=ind)
            df_out_p = pd.concat([df_msk, df_prob], axis = 1).drop(columns=['mask'])
            prob = df_out_p['probability'].tolist()
            
        else:
            prob = np.repeat([[np.nan]]*msk.height, msk.width,axis=1).tolist()
            
        # Reshape the probability raster
        class_prob = np.array(prob).reshape(img.shape)
        
        # Create output file name
        imgname = os.path.basename(filename).split('.')[0]
        path = dir_class + imgname + "_" + modelname + "_prob.tif"
        
        # get image metadata
        kwargs = msk.meta
        
        # save output rasters
        with rasterio.open(path, 'w', **kwargs) as dst:
            dst.write_band(1, class_prob)
    
        print(path)

## Mosaic image tiles after processing

- Creates a mosaic of image tiles post-classification (if required)
- Change the folder paths to where tiles to mosaic are saved, and set output mosaic file path/name

In [ ]:
import rasterio
from rasterio.merge import merge
from rasterio.plot import show

#Set directories
dirpath = r"..."
out_fp = r"...\mosaic.tif"

count = 0
for path in os.listdir(dirpath):
    if os.path.isfile(os.path.join(dirpath, path)):
        count += 1
print("Images to mosaic = " + str(count))

#Empty list for the datafiles that will be part of the mosaic
src_files_to_mosaic = []

#Open files in read only mode with rasterio and add those files to the source file list
for filename in glob.iglob(f'{dirpath}/*'):
    src = rasterio.open(filename)
    src_files_to_mosaic.append(src)

#merge function returns a single mosaic array and the transformation info
mosaic, out_trans = merge(src_files_to_mosaic)
show(mosaic)

# Copy the metadata
out_meta = src.meta.copy()

# Update the metadata
out_meta.update({"driver": "GTiff",
                 "height": mosaic.shape[1],
                 "width": mosaic.shape[2],
                 "transform": out_trans,
                 "crs":src.crs
                 }
                )

with rasterio.open(out_fp, "w", **out_meta) as dest:
    dest.write(mosaic)

print("done")